# Results

Loads the two committed training runs and reproduces the figure and the table
in the README. Nothing here retrains anything.

`train.py` writes one JSON object per evaluation step, so each run is a
`.jsonl` file under `logs/`.

In [ ]:
# This notebook lives in notebooks/, but config.py and logs/ are at the repo
# root - so work from there regardless of where Jupyter was launched.
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(REPO_ROOT)
sys.path.insert(0, os.getcwd())

import glob
import json

import numpy as np
import matplotlib.pyplot as plt

from config import log_dir

FIG_PATH = os.path.join("figures", "loss_curves.png")
os.makedirs("figures", exist_ok=True)
print("working from:", os.getcwd())

In [ ]:
def load_run(pattern):
    """Load the most recent .jsonl log matching `pattern`.

    Each line is one eval step: {"step", "train_loss", "val_loss", "is_best"}.
    """
    files = sorted(glob.glob(os.path.join(log_dir, pattern)), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(f"No logs matching {pattern!r} in {log_dir}/")
    path = files[-1]
    with open(path, encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    print(f"{path}  ({len(records)} eval points)")
    return {
        "steps": [r["step"] for r in records],
        "train": [r["train_loss"] for r in records],
        "val":   [r["val_loss"] for r in records],
        "best":  [r["is_best"] for r in records],
    }


cont  = load_run("train_tail_*.jsonl")    # contiguous: last 10% held out
chunk = load_run("train_chunk_*.jsonl")   # chunk-shuffled

In [ ]:
def summarise(run, name):
    gap = np.array(run["val"]) - np.array(run["train"])
    last10 = run["best"][-10:]
    return {
        "split": name,
        "best val loss": min(run["val"]),
        "final train loss": run["train"][-1],
        "final val loss": run["val"][-1],
        "train/val gap": gap[-1],
        "val improved in last 10 evals": f"{sum(last10)} / {len(last10)}",
    }


rows = [summarise(cont, "contiguous"), summarise(chunk, "chunk-shuffled")]

print(f"{'':<32} {'contiguous':>12} {'chunk-shuffled':>15}")
for k in rows[0]:
    if k == "split":
        continue
    a, b = rows[0][k], rows[1][k]
    fmt = (lambda v: f"{v:.4f}") if isinstance(a, float) else str
    print(f"{k:<32} {fmt(a):>12} {fmt(b):>15}")

## The figure

Left: the loss curves. Chunk-shuffled val sits lower, which looks like the
better run. Note that the two dashed train curves nearly overlap — the models
are equally good.

Right: the gap between val and train. This is where the two runs actually
differ.

In [ ]:
C_CONT, C_CHUNK = "#C44E52", "#4C72B0"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

ax1.plot(cont["steps"],  cont["train"],  color=C_CONT,  ls="--", lw=1.4, alpha=.75,
         label="contiguous — train")
ax1.plot(cont["steps"],  cont["val"],    color=C_CONT,  lw=2.2, label="contiguous — val")
ax1.plot(chunk["steps"], chunk["train"], color=C_CHUNK, ls="--", lw=1.4, alpha=.75,
         label="chunk-shuffled — train")
ax1.plot(chunk["steps"], chunk["val"],   color=C_CHUNK, lw=2.2, label="chunk-shuffled — val")

ax1.set_xlabel("step"); ax1.set_ylabel("cross-entropy loss")
ax1.set_title("Same model, two train/val splits", fontsize=12, fontweight="bold")
ax1.set_ylim(min(cont["train"] + chunk["train"]) - 0.05, 2.6)
ax1.legend(fontsize=9, framealpha=.9)
ax1.grid(alpha=.25, ls=":")
ax1.spines[["top", "right"]].set_visible(False)

gap_cont  = np.array(cont["val"])  - np.array(cont["train"])
gap_chunk = np.array(chunk["val"]) - np.array(chunk["train"])

ax2.plot(cont["steps"],  gap_cont,  color=C_CONT,  lw=2.2, label="contiguous")
ax2.plot(chunk["steps"], gap_chunk, color=C_CHUNK, lw=2.2, label="chunk-shuffled")
ax2.fill_between(cont["steps"], gap_chunk, gap_cont, color="grey", alpha=.12)
ax2.axhline(0, color="k", lw=.7, alpha=.4)

ax2.set_xlabel("step"); ax2.set_ylabel("val loss − train loss")
ax2.set_title(f"Generalization gap ({gap_cont[-1] / gap_chunk[-1]:.1f}× smaller "
              f"when val leaks context)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9, framealpha=.9)
ax2.grid(alpha=.25, ls=":")
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle("Tiny Shakespeare: the better val loss is the easier validation set",
             fontsize=13.5, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIG_PATH, dpi=160)
print("wrote", FIG_PATH)